# CoSwitchNLP — Exploratory Data Analysis

This notebook walks through the SentiMix (SemEval-2020 Task 9) dataset to understand:
- Class distributions for language ID tags and sentiment labels
- Token-level language mixing patterns
- Code-Mixing Index (CMI) distribution
- Example sentences and their annotations

**Run the backend `train.py` first, or at minimum download the dataset to `../data/sentimix/`.**

In [ ]:
import sys
sys.path.insert(0, '../backend')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

from dataset import load_sentimix_conll
from inference import compute_cmi

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Imports OK')

In [ ]:
# Load data
train = load_sentimix_conll('../data/sentimix/train.txt')
val   = load_sentimix_conll('../data/sentimix/val.txt')

print(f'Train: {len(train)} sentences')
print(f'Val:   {len(val)} sentences')
print()
print('Sample:')
ex = train[0]
for tok, tag in zip(ex['tokens'], ex['lang_tags']):
    print(f'  {tok:20s} {tag}')
print(f'  Sentiment: {ex["sentiment"]}')

## Sentiment Label Distribution

In [ ]:
sent_counts = Counter(ex['sentiment'] for ex in train)
print('Sentiment distribution (train):')
total = sum(sent_counts.values())
for label, count in sorted(sent_counts.items()):
    print(f'  {label:10s}: {count:5d} ({count/total*100:.1f}%)')

fig, ax = plt.subplots(figsize=(5, 3.5))
colors = {'positive': '#22c55e', 'neutral': '#f59e0b', 'negative': '#ef4444'}
labels = list(sent_counts.keys())
values = [sent_counts[l] for l in labels]
bars = ax.bar(labels, values, color=[colors.get(l, '#6b7280') for l in labels], edgecolor='white', linewidth=0.8)
ax.bar_label(bars, fmt='%d', padding=3, fontsize=10)
ax.set_title('Sentiment Label Distribution (Train)', fontsize=12, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## Language Tag Distribution

In [ ]:
tag_counts = Counter()
for ex in train:
    tag_counts.update(ex['lang_tags'])

print('LID tag distribution (train tokens):')
total_tokens = sum(tag_counts.values())
for tag, count in tag_counts.most_common():
    print(f'  {tag:10s}: {count:7d} ({count/total_tokens*100:.1f}%)')

tag_colors = {
    'lang1': '#f97316', 'lang2': '#3b82f6', 'mixed': '#a855f7',
    'ne': '#22c55e', 'other': '#9ca3af', 'univ': '#d1d5db'
}
fig, ax = plt.subplots(figsize=(6, 3.5))
sorted_tags = tag_counts.most_common()
labels = [t[0] for t in sorted_tags]
values = [t[1] for t in sorted_tags]
bars = ax.bar(labels, values, color=[tag_colors.get(l, '#9ca3af') for l in labels])
ax.bar_label(bars, fmt='%d', padding=3, fontsize=9)
ax.set_title('Language Tag Distribution (Train Tokens)', fontsize=12, fontweight='bold')
ax.set_ylabel('Token count')
plt.tight_layout()
plt.show()

## Code-Mixing Index Distribution

In [ ]:
import numpy as np

cmis = [compute_cmi(ex['lang_tags']) for ex in train]
print(f'CMI stats:')
print(f'  Mean:   {np.mean(cmis):.3f}')
print(f'  Median: {np.median(cmis):.3f}')
print(f'  Std:    {np.std(cmis):.3f}')
print(f'  Min:    {np.min(cmis):.3f}')
print(f'  Max:    {np.max(cmis):.3f}')

pct_zero = sum(1 for c in cmis if c == 0) / len(cmis) * 100
pct_high = sum(1 for c in cmis if c > 0.5) / len(cmis) * 100
print(f'  Monolingual (CMI=0): {pct_zero:.1f}%')
print(f'  Highly mixed (CMI>0.5): {pct_high:.1f}%')

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(cmis, bins=30, color='#6366f1', edgecolor='white', linewidth=0.5)
ax.axvline(np.mean(cmis), color='#f97316', linestyle='--', linewidth=1.5, label=f'Mean={np.mean(cmis):.2f}')
ax.set_title('Code-Mixing Index Distribution (Train)', fontsize=12, fontweight='bold')
ax.set_xlabel('CMI')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

## Sentence Length Distribution

In [ ]:
lengths = [len(ex['tokens']) for ex in train]
print(f'Sentence length: mean={np.mean(lengths):.1f}, median={np.median(lengths):.0f}, max={max(lengths)}')

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(lengths, bins=40, color='#06b6d4', edgecolor='white', linewidth=0.5)
ax.axvline(np.mean(lengths), color='#f97316', linestyle='--', linewidth=1.5, label=f'Mean={np.mean(lengths):.1f}')
ax.set_title('Sentence Length Distribution (Train)', fontsize=12, fontweight='bold')
ax.set_xlabel('Tokens per sentence')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

## Sample Annotated Sentences

In [ ]:
TAG_COLORS_ANSI = {
    'lang1': '\033[93m',  # Yellow
    'lang2': '\033[94m',  # Blue
    'mixed': '\033[95m',  # Magenta
    'ne':    '\033[92m',  # Green
    'other': '\033[90m',  # Gray
    'univ':  '\033[90m',
}
RESET = '\033[0m'

def display_sentence(ex):
    print(f"Sentiment: {ex['sentiment'].upper()}")
    print(f"CMI: {compute_cmi(ex['lang_tags']):.2f}")
    parts = []
    for tok, tag in zip(ex['tokens'], ex['lang_tags']):
        color = TAG_COLORS_ANSI.get(tag, '')
        parts.append(f"{color}{tok}[{tag}]{RESET}")
    print(' '.join(parts))
    print()

# Show 5 examples from each sentiment class
for sentiment in ['positive', 'negative', 'neutral']:
    print(f"{'='*60}")
    print(f"  {sentiment.upper()} examples")
    print(f"{'='*60}")
    shown = 0
    for ex in train:
        if ex['sentiment'] == sentiment:
            display_sentence(ex)
            shown += 1
            if shown >= 3:
                break

## Training History (after training)

In [ ]:
import json, os

history_path = '../models/coswitchnlp_v1/training_history.json'
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)

    epochs = [h['epoch'] for h in history]
    train_losses = [h['train_loss'] for h in history]
    val_losses = [h['loss'] for h in history]
    sent_f1s = [h['sentiment_f1'] for h in history]
    lid_f1s = [h['lid_f1'] for h in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    ax1.plot(epochs, train_losses, 'o-', label='Train loss', color='#6366f1')
    ax1.plot(epochs, val_losses, 's--', label='Val loss', color='#f97316')
    ax1.set_title('Loss', fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.legend()

    ax2.plot(epochs, sent_f1s, 'o-', label='Sentiment F1', color='#22c55e')
    ax2.plot(epochs, lid_f1s, 's--', label='LID F1', color='#3b82f6')
    ax2.set_title('Validation F1', fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.legend()

    plt.suptitle('Training History', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print(f'Training history not found at {history_path}.')
    print('Run train.py first.')